In [1]:
import gc
import tensorflow as tf
from keras import backend as K

K.clear_session()
gc.collect()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

In [2]:
import ast
from pathlib import Path
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import keras


from sklearn.metrics import f1_score, multilabel_confusion_matrix, roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer

from tensorflow.keras import mixed_precision, layers, Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, ReLU, Add, MaxPooling1D,
    GlobalAveragePooling1D, Dense, Dropout, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [3]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / "X_train_processed.npy").exists():
    BASE_DIR = Path(r"D:\AOU-Graduation-Project\BioIntellect\AI\ECG\App\classification")

TRAIN_AUDIO_PATH = BASE_DIR / "X_train_processed.npy"
VAL_AUDIO_PATH = BASE_DIR / "X_val_processed.npy"
TEST_AUDIO_PATH = BASE_DIR / "X_test_processed.npy"

TRAIN_CSV_PATH = BASE_DIR / "ptbxl_train.csv"
VAL_CSV_PATH = BASE_DIR / "ptbxl_val.csv"
TEST_CSV_PATH = BASE_DIR / "ptbxl_test.csv"

CSV_FEATURE_COLUMNS = [
    "age",
    "sex",
    "height",
    "weight",
    "validated_by_human",
    "height_missing",
    "weight_missing",
]

EXPECTED_SIGNAL_SHAPE = (5000, 12)
BATCH_SIZE = 32
PREDICT_BATCH_SIZE = 6     
DOWNSAMPLE_STRIDE=5   
EPOCHS = 30
LEARNING_RATE = 3e-4
METRIC_THRESHOLD = 0.5
THRESHOLD_SEARCH_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
POS_WEIGHTS_PATH = BASE_DIR / "pos_weights.npy"
USE_POS_WEIGHTS = False
POS_WEIGHTS_MAX_CAP = 10.0   

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

policy_name = "mixed_float16" if gpus else "float32"
mixed_precision.set_global_policy(policy_name)

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU, compute capability 8.6


In [4]:
def load_signal_npy(npy_path, mmap_mode="r"):
    signals = np.load(npy_path, allow_pickle=False, mmap_mode=mmap_mode)

    if signals.dtype != np.float32:
        signals = signals.astype(np.float32)

    if signals.ndim != 3 or tuple(signals.shape[1:]) != EXPECTED_SIGNAL_SHAPE:
        raise ValueError(
            f"Expected shape (N, {EXPECTED_SIGNAL_SHAPE[0]}, {EXPECTED_SIGNAL_SHAPE[1]}), got {signals.shape}"
        )

    return signals


def extract_codes_list(df):
    return df["scp_codes"].apply(lambda value: sorted(ast.literal_eval(value).keys()))


def build_feature_matrix(df, feature_cols):
    features = df[feature_cols].copy()

    bool_cols = features.select_dtypes(include=["bool"]).columns
    if len(bool_cols):
        features[bool_cols] = features[bool_cols].astype(np.float32)

    return features.astype(np.float32).to_numpy()


def assert_split_alignment(signals, features, labels, split_name):
    if len(signals) != len(features):
        raise ValueError(
            f"{split_name}: signals/features mismatch -> {len(signals)} vs {len(features)}"
        )

    if labels is not None and len(labels) != len(signals):
        raise ValueError(
            f"{split_name}: labels/signals mismatch -> {len(labels)} vs {len(signals)}"
        )


In [5]:
def asymmetric_loss(gamma_neg=4, gamma_pos=0, clip=0.05):
    """
    Ridnik et al. ICCV 2021
    gamma_neg=4, gamma_pos=0 هي القيم الموصى بها في الورقة
    clip=0.05 بيمنع الـ easy negatives من تخريب الـ training
    """
    def asl(y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        
        # Asymmetric clipping
        y_pred_neg = tf.clip_by_value(y_pred + clip, 0, 1)
        
        # Probability margins
        xs_pos = y_pred
        xs_neg = 1 - y_pred_neg
        
        # Log probabilities
        los_pos = y_true * tf.math.log(tf.clip_by_value(xs_pos, 1e-7, 1))
        los_neg = (1 - y_true) * tf.math.log(tf.clip_by_value(xs_neg, 1e-7, 1))
        
        loss = los_pos + los_neg
        
        # Asymmetric focusing
        if gamma_neg > 0 or gamma_pos > 0:
            pt0 = xs_pos * y_true
            pt1 = xs_neg * (1 - y_true)
            pt = pt0 + pt1
            
            one_sided_gamma = gamma_pos * y_true + gamma_neg * (1 - y_true)
            one_sided_w = tf.pow(1 - pt, one_sided_gamma)
            loss *= one_sided_w
        
        return -tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    
    asl.__name__ = "asymmetric_loss"
    return asl


In [6]:
def build_multimodal_head(
    backbone,
    signal_shape,
    num_csv_features,
    num_classes,
    learning_rate=LEARNING_RATE,
    model_name="Multimodal_ECG",
):
    """
    Generic multimodal head — يشتغل مع InceptionTime أو XceptionTime.
    backbone: الموديل اللي بترجعه من build_*_backbone()
    """
    signal_input = Input(shape=signal_shape, dtype=tf.float32, name="signal_input")
    csv_input    = Input(shape=(num_csv_features,), dtype=tf.float32, name="csv_input")

    # Signal branch
    signal_features = backbone(signal_input)

    # CSV branch
    csv_x = Dense(64, activation="relu", dtype="float32", name="csv_dense1")(csv_input)
    csv_x = Dropout(0.15, name="csv_dropout1")(csv_x)
    csv_x = Dense(32, activation="relu", dtype="float32", name="csv_dense2")(csv_x)

    # Fusion
    merged = Concatenate(name="fusion")([signal_features, csv_x])

    # Classification head
    x = Dense(256, activation="relu", dtype="float32", name="head_dense1")(merged)
    x = Dropout(0.30, name="head_dropout1")(x)
    x = Dense(128, activation="relu", dtype="float32", name="head_dense2")(x)
    x = Dropout(0.20, name="head_dropout2")(x)
    output = Dense(num_classes, activation="sigmoid", dtype="float32", name="output")(x)

    model = Model(inputs=[signal_input, csv_input], outputs=output, name=model_name)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=asymmetric_loss(gamma_neg=4, gamma_pos=0, clip=0.05),        
        metrics=[
            tf.keras.metrics.AUC(name="auc_roc", multi_label=True),
            tf.keras.metrics.AUC(name="pr_auc", multi_label=True, curve="PR"),
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    model.summary()
    return model


In [7]:
def _inception_module(x, nb_filters=32, kernel_sizes=(10, 20, 40), bottleneck_size=32, name="inception"):
    """
    القلب النابض في InceptionTime.
    
    الفكرة الأساسية: بدل ما نشوف الـ ECG بعين واحدة (kernel واحد)،
    بنشوفه بـ 4 عيون في نفس الوقت — كل عين بتشوف نطاق زمني مختلف.
    
    ليه؟ لأن الـ ECG events بيتفاوتوا في الحجم:
      - موجة P  →  صغيرة وسريعة  →  kernel صغير يمسكها
      - QRS complex  →  متوسطة  →  kernel متوسط
      - موجة T  →  كبيرة وبطيئة  →  kernel كبير
    
    الـ flow:
        x
        ├─ Bottleneck (تضغط الـ channels لتوفير VRAM)
        │   ├─ Conv(k=10)  →  branch_1
        │   ├─ Conv(k=20)  →  branch_2  
        │   └─ Conv(k=40)  →  branch_3
        └─ MaxPool(3)  →  Conv(1)  →  branch_4
        
        [branch_1, branch_2, branch_3, branch_4]
        → Concatenate → BatchNorm → ReLU → output
    """

    in_channels = x.shape[-1]  # عدد الـ channels الداخلة (مثلاً 32 أو 128)

    # ── Step 1: Bottleneck ──────────────────────────────────────────────────────
    # المشكلة: لو عندنا 128 channel وعايزين نعمل Conv(k=40) عليهم
    #          ده هيكلف VRAM كتير جداً.
    # الحل: نضغط الـ channels لـ 32 الأول بـ Conv(1×1) (مش بتغير الـ time steps)
    #       وبعدين نعمل الـ parallel convs على الـ 32 الأصغر.
    # ملحوظة: لو الـ channels أصغر من bottleneck_size أصلاً → مش محتاجين نضغط
    if bottleneck_size is not None and in_channels is not None and in_channels > bottleneck_size:
        x_bottleneck = layers.Conv1D(
            bottleneck_size,    # نضغط لـ 32 channel
            kernel_size=1,      # 1×1 conv → بس بيغير الـ channels مش الـ time
            padding='same',
            use_bias=False,     # BatchNorm جاي بعدين، الـ bias مش مفيد هنا
            name=f"{name}_bottleneck"
        )(x)
    else:
        x_bottleneck = x  # خليها زي ما هي

    # ── Step 2: Parallel Convolutions (3 branches) ─────────────────────────────
    # كل branch بيشوف نطاق زمني مختلف من الـ signal
    branches = []
    for k in kernel_sizes:  # k = 10, 20, 40
        branch = layers.Conv1D(
            nb_filters,     # 32 filter لكل branch
            kernel_size=k,  # النافذة الزمنية
            padding='same', # نحافظ على نفس عدد الـ time steps
            use_bias=False,
            name=f"{name}_conv_k{k}"
        )(x_bottleneck)
        branches.append(branch)

    # ── Step 3: MaxPool Branch ──────────────────────────────────────────────────
    # الـ branch ده مش بيتعلم — بس بياخد أقوى قيمة في كل نافذة صغيرة.
    # بيمسك الـ peaks الواضحة في الـ signal (زي الـ R peak في QRS).
    # مهم: بيشتغل على x الأصلي (مش الـ bottleneck) عشان ميخسرش معلومات.
    mp = layers.MaxPooling1D(
        pool_size=3,    # بيشوف 3 نقاط ويرجع الأكبر
        strides=1,      # بيتحرك نقطة نقطة (مش بيقلل الـ time steps)
        padding='same',
        name=f"{name}_maxpool"
    )(x)
    
    # Conv(1×1) بعد الـ MaxPool عشان يضبط عدد الـ channels زي باقي الـ branches
    mp = layers.Conv1D(
        nb_filters,
        kernel_size=1,
        padding='same',
        use_bias=False,
        name=f"{name}_mp_conv"
    )(mp)
    
    branches.append(mp)  # دلوقتي عندنا 4 branches

    # ── Step 4: Merge ───────────────────────────────────────────────────────────
    # بنجمع الـ 4 branches جنب بعض على الـ channel axis
    # لو كل branch فيه 32 channel → بعد الـ concat هيبقى 32×4 = 128 channel
    out = layers.Concatenate(name=f"{name}_concat")(branches)
    
    # BatchNorm: بينظم الـ values ويسرع الـ training
    out = layers.BatchNormalization(name=f"{name}_bn")(out)
    
    # ReLU: بيعمل non-linearity (بيحول الـ negative values لـ 0)
    out = layers.ReLU(name=f"{name}_relu")(out)
    
    return out  # shape: (batch, time_steps, 128)


def build_inceptiontime_backbone(
    input_shape=(5000, 12),   # 5 ثواني × 500Hz × 12 lead
    nb_filters=32,            # filters في كل branch داخل الـ inception
    depth=6,                  # عدد الـ inception modules (6 في البيبر الأصلي)
    kernel_sizes=(10, 20, 40),
    bottleneck_size=32,
    downsample_stride=5,      # 5 → بيقلل 5000 timestep لـ 1000
):
    """
    الـ backbone كله — بياخد ECG signal خام ويرجع feature vector جاهز للتصنيف.
    
    الـ flow الكامل:
    
    Input (batch, 5000, 12)
         │
         ▼
    [Downsample] Conv1D(stride=5) → BN → ReLU
    (batch, 1000, 32)
         │
         ├─────────────────── residual_0 = x
         │
         ▼
    Inception_0  →  Inception_1  →  Inception_2
                                         │
                                    Add(x + shortcut)  ← Conv1D(residual_0)
                                    residual_1 = x
         │
         ▼
    Inception_3  →  Inception_4  →  Inception_5
                                         │
                                    Add(x + shortcut)  ← Conv1D(residual_1)
         │
         ▼
    GlobalAveragePooling1D
    (batch, 128)  ← feature vector جاهز
    """

    inputs = layers.Input(shape=input_shape, name="ecg_input")

    # ── Step 1: Downsampling ────────────────────────────────────────────────────
    # المشكلة: 5000 timestep كتير جداً على الـ GPU (خصوصاً مع batch_size صغير).
    # الحل: Conv1D بـ stride=5 تبقى 1000 timestep — نفس المعلومات تقريباً، VRAM أقل بكتير.
    # kernel_size = stride*2 = 10 عشان يشوف نافذة كافية قبل ما يقفز.
    if downsample_stride > 1:
        x = layers.Conv1D(
            nb_filters,
            kernel_size=downsample_stride * 2,
            strides=downsample_stride,
            padding='same',
            use_bias=False,
            name="initial_downsample"
        )(inputs)
        x = layers.BatchNormalization(name="initial_downsample_bn")(x)
        x = layers.ReLU(name="initial_downsample_relu")(x)
    else:
        x = inputs  # مفيش downsample → ابدأ من 5000 مباشرةً

    # ── Step 2: Residual Anchor ─────────────────────────────────────────────────
    # بنحفظ نسخة من x الحالي قبل ما ندخل الـ inception blocks.
    # هنستخدمها كـ "shortcut" بعد كل 3 blocks (زي ResNet).
    # الهدف: لو الـ inception blocks اتعلموا حاجة غلط → الـ gradient يعدي
    #         عن طريق الـ shortcut بدل ما يـ vanish.
    residual = x

    # عدد الـ output channels من كل inception module:
    # 3 parallel convs + 1 maxpool branch = 4 branches × 32 filter = 128
    n_out_channels = nb_filters * (len(kernel_sizes) + 1)

    # ── Step 3: Inception Blocks ────────────────────────────────────────────────
    for i in range(depth):  # i = 0, 1, 2, 3, 4, 5
        x = _inception_module(
            x,
            nb_filters=nb_filters,
            kernel_sizes=kernel_sizes,
            bottleneck_size=bottleneck_size,
            name=f"inception_{i}"
        )

        # كل 3 blocks → residual shortcut
        # i=2 → (2+1)%3 == 0 ✓  →  أول shortcut
        # i=5 → (5+1)%3 == 0 ✓  →  تاني shortcut
        if (i + 1) % 3 == 0:
            # الـ residual ممكن channels-ها تختلف عن x الحالي
            # فبنعمل Conv(1×1) عشان نوحد الـ channels قبل الـ Add
            shortcut = layers.Conv1D(
                n_out_channels,
                kernel_size=1,
                padding='same',
                use_bias=False,
                name=f"res_conv_{i}"
            )(residual)
            shortcut = layers.BatchNormalization(name=f"res_bn_{i}")(shortcut)

            # بنضيف الـ shortcut على x الحالي (مش concatenate — Add)
            x = layers.Add(name=f"res_add_{i}")([x, shortcut])
            x = layers.ReLU(name=f"res_relu_{i}")(x)

            # نحدث الـ residual للـ shortcut الجاي
            residual = x

    # ── Step 4: Global Average Pooling ─────────────────────────────────────────
    # دلوقتي x شكله (batch, 1000, 128).
    # الـ classifier مش محتاج يعرف "فين" الـ feature في الـ time axis،
    # محتاج بس "إيه" الـ features الموجودة → نعمل average على كل الـ time steps.
    # النتيجة: (batch, 128) — vector واحد يمثل الـ ECG كله.
    x = GlobalAveragePooling1D(name="gap")(x)

    return Model(inputs=inputs, outputs=x, name="InceptionTime_Backbone")

def build_multimodal_inceptiontime(
    signal_shape,
    num_csv_features,
    num_classes,
    learning_rate=LEARNING_RATE,
    nb_filters=32,
    depth=6,
    downsample_stride=DOWNSAMPLE_STRIDE,
):

    backbone = build_inceptiontime_backbone(
        input_shape=signal_shape,
        nb_filters=nb_filters,
        depth=depth,
        downsample_stride=downsample_stride,
    )

    return build_multimodal_head(
        backbone=backbone,
        signal_shape=signal_shape,
        num_csv_features=num_csv_features,
        num_classes=num_classes,
        learning_rate=learning_rate,
        model_name="Multimodal_InceptionTime",
    )

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
val_df = pd.read_csv(VAL_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

X_train_signal = load_signal_npy(TRAIN_AUDIO_PATH)
X_val_signal = load_signal_npy(VAL_AUDIO_PATH)
X_test_signal = load_signal_npy(TEST_AUDIO_PATH)

X_train_csv = build_feature_matrix(train_df, CSV_FEATURE_COLUMNS)
X_val_csv = build_feature_matrix(val_df, CSV_FEATURE_COLUMNS)
X_test_csv = build_feature_matrix(test_df, CSV_FEATURE_COLUMNS)

mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(extract_codes_list(train_df)).astype(np.float32)
y_val = mlb.transform(extract_codes_list(val_df)).astype(np.float32)
y_test = mlb.transform(extract_codes_list(test_df)).astype(np.float32)

assert_split_alignment(X_train_signal, X_train_csv, y_train, "train")
assert_split_alignment(X_val_signal, X_val_csv, y_val, "val")
assert_split_alignment(X_test_signal, X_test_csv, y_test, "test")

SIGNAL_SHAPE = X_train_signal.shape[1:]
NUM_CLASSES = y_train.shape[1]

print("Train signal:", X_train_signal.shape, X_train_signal.dtype)
print("Val signal:", X_val_signal.shape, X_val_signal.dtype)
print("Test signal:", X_test_signal.shape, X_test_signal.dtype)
print("Train CSV:", X_train_csv.shape, X_train_csv.dtype)
print("Labels:", NUM_CLASSES)
print("classes:", mlb.classes_[:].tolist())

Train signal: (17418, 5000, 12) float32
Val signal: (2183, 5000, 12) float32
Test signal: (2198, 5000, 12) float32
Train CSV: (17418, 7) float32
Labels: 71
classes: ['1AVB', '2AVB', '3AVB', 'ABQRS', 'AFIB', 'AFLT', 'ALMI', 'AMI', 'ANEUR', 'ASMI', 'BIGU', 'CLBBB', 'CRBBB', 'DIG', 'EL', 'HVOLT', 'ILBBB', 'ILMI', 'IMI', 'INJAL', 'INJAS', 'INJIL', 'INJIN', 'INJLA', 'INVT', 'IPLMI', 'IPMI', 'IRBBB', 'ISCAL', 'ISCAN', 'ISCAS', 'ISCIL', 'ISCIN', 'ISCLA', 'ISC_', 'IVCD', 'LAFB', 'LAO/LAE', 'LMI', 'LNGQT', 'LOWT', 'LPFB', 'LPR', 'LVH', 'LVOLT', 'NDT', 'NORM', 'NST_', 'NT_', 'PAC', 'PACE', 'PMI', 'PRC(S)', 'PSVT', 'PVC', 'QWAVE', 'RAO/RAE', 'RVH', 'SARRH', 'SBRAD', 'SEHYP', 'SR', 'STACH', 'STD_', 'STE_', 'SVARR', 'SVTAC', 'TAB_', 'TRIGU', 'VCLVH', 'WPW']


In [9]:
model = build_multimodal_inceptiontime(
    signal_shape=SIGNAL_SHAPE,
    num_csv_features=X_train_csv.shape[1],
    num_classes=NUM_CLASSES,
    nb_filters=32,
    depth=6,
    downsample_stride=DOWNSAMPLE_STRIDE,
)

model.summary()

Model: "Multimodal_InceptionTime"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 csv_input (InputLayer)         [(None, 7)]          0           []                               
                                                                                                  
 csv_dense1 (Dense)             (None, 64)           512         ['csv_input[0][0]']              
                                                                                                  
 signal_input (InputLayer)      [(None, 5000, 12)]   0           []                               
                                                                                                  
 csv_dropout1 (Dropout)         (None, 64)           0           ['csv_dense1[0][0]']             
                                                                           

In [10]:
class MacroF1Callback(tf.keras.callbacks.Callback):
    """
    بيحسب F1 macro على الـ validation set بعد كل epoch.
    أسرع من sklearn لأنه بيشتغل بـ numpy مباشرة.
    """
    def __init__(self, val_data, threshold=0.5):
        super().__init__()
        self.X_val_signal, self.X_val_csv = val_data[0]
        self.y_val = val_data[1]
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        predictions = []
        for start in range(0, len(self.X_val_signal), PREDICT_BATCH_SIZE):
            end = min(start + PREDICT_BATCH_SIZE, len(self.X_val_signal))
            batch_pred = self.model(
                [self.X_val_signal[start:end], self.X_val_csv[start:end]],
                training=False
            ).numpy()
            predictions.append(batch_pred)
            gc.collect()
        
        y_pred = np.vstack(predictions).astype(np.float32)
        y_pred_bin = (y_pred >= self.threshold).astype(np.float32)
        y_true = self.y_val

        # F1 per class ثم average — نفس منطق sklearn
        tp = (y_true * y_pred_bin).sum(axis=0)
        fp = ((1 - y_true) * y_pred_bin).sum(axis=0)
        fn = (y_true * (1 - y_pred_bin)).sum(axis=0)

        precision = tp / np.clip(tp + fp, 1e-7, None)
        recall    = tp / np.clip(tp + fn, 1e-7, None)
        f1_per_class = 2 * precision * recall / np.clip(precision + recall, 1e-7, None)

        # نتجاهل الـ classes اللي مفيش samples منها في الـ val
        valid_mask = y_true.sum(axis=0) > 0
        f1_macro = f1_per_class[valid_mask].mean()

        logs["val_f1_macro"] = float(f1_macro)
        print(f"  — val_f1_macro: {f1_macro:.4f}")



macro_f1_cb = MacroF1Callback(
    val_data=([X_val_signal, X_val_csv], y_val),
    threshold=0.5,
)

callbacks = [
    macro_f1_cb, 

    ModelCheckpoint(
        BASE_DIR / "ecg_inceptiontime_multimodal_asymmetric_loss_loss-min_best.h5",
        monitor="val_loss",  
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),

    EarlyStopping(
        monitor="val_loss",   
        mode="min",
        patience=10,              
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",   
        factor=0.5,
        patience=4,               
        mode="min",
        min_lr=1e-6,           
        verbose=1,
    ),
]

In [ ]:
history = model.fit(
    [X_train_signal, X_train_csv],
    y_train,
    validation_data=([X_val_signal, X_val_csv], y_val),
    epochs=100,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)



Epoch 1/100
545/545 [==============================] - ETA: 0s - loss: 2.7600 - auc_roc: 0.6805 - pr_auc: 0.0834 - binary_accuracy: 0.9532 - precision: 0.4253 - recall: 0.5289  — val_f1_macro: 0.1042

Epoch 1: val_loss improved from inf to 2.38973, saving model to d:\AOU-Graduation-Project\BioIntellect\AI\ECG\App\classification\ecg_inceptiontime_multimodal_asymmetric_loss_loss-min_best.h5
545/545 [==============================] - 98s 156ms/step - loss: 2.7600 - auc_roc: 0.6805 - pr_auc: 0.0834 - binary_accuracy: 0.9532 - precision: 0.4253 - recall: 0.5289 - val_loss: 2.3897 - val_auc_roc: 0.7867 - val_pr_auc: 0.1669 - val_binary_accuracy: 0.9731 - val_precision: 0.7104 - val_recall: 0.5298 - val_f1_macro: 0.1042 - lr: 3.0000e-04
Epoch 2/100
544/545 [============================>.] - ETA: 0s - loss: 2.2055 - auc_roc: 0.7952 - pr_auc: 0.1698 - binary_accuracy: 0.9717 - precision: 0.6461 - recall: 0.6247  — val_f1_macro: 0.1584

Epoch 2: val_loss improved from 2.38973 to 2.12668, saving 

In [12]:
def summarize_history(history):
    history_df = pd.DataFrame(history.history)
    print("Last epoch metrics:")
    print(history_df.tail(1).round(4).to_string(index=False))

    if "val_auc_roc" in history_df.columns:
        best_epoch = int(history_df["val_auc_roc"].idxmax()) + 1

        best_val_auc = float(history_df["val_auc_roc"].max())
        print(f"Best validation AUC epoch: {best_epoch}")
        print(f"Best validation AUC: {best_val_auc:.4f}")

    return history_df


def plot_history(history_df):
    if history_df.empty:
        print("No history to plot.")
        return None

    epochs = range(1, len(history_df) + 1)
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

    if "loss" in history_df.columns:
        axes[0].plot(epochs, history_df["loss"], marker="o", label="loss")
    if "val_loss" in history_df.columns:
        axes[0].plot(epochs, history_df["val_loss"], marker="o", label="val_loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if "binary_accuracy" in history_df.columns:
        axes[1].plot(epochs, history_df["binary_accuracy"], marker="o", label="binary_accuracy")
    if "val_binary_accuracy" in history_df.columns:
        axes[1].plot(epochs, history_df["val_binary_accuracy"], marker="o", label="val_binary_accuracy")
    axes[1].set_title("Binary Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# في plot_history
    if "auc_roc" in history_df.columns:
        axes[2].plot(epochs, history_df["auc_roc"], label="auc_roc")
    if "val_auc_roc" in history_df.columns:
        axes[2].plot(epochs, history_df["val_auc_roc"], label="val_auc_roc")
        
    axes[2].set_title("AUC")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig



def safe_multilabel_auc(y_true, y_score, average):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    valid_columns = []
    for i in range(y_true.shape[1]):
        positive_count = y_true[:, i].sum()
        if 0 < positive_count < len(y_true):
            valid_columns.append(i)

    if not valid_columns:
        return np.nan

    return float(roc_auc_score(y_true[:, valid_columns], y_score[:, valid_columns], average=average))


def batched_multimodal_predict(model, x_signal, x_csv, batch_size=None):
    """بتقييم الموديل على batches صغيرة لتجنب MemoryError"""
    if batch_size is None:
        batch_size = PREDICT_BATCH_SIZE   # <<< استخدام batch أصغر

    if len(x_signal) != len(x_csv):
        raise ValueError(f"signals/features mismatch -> {len(x_signal)} vs {len(x_csv)}")

    predictions = []
    for start in range(0, len(x_signal), batch_size):
        end = min(start + batch_size, len(x_signal))
        signal_batch = np.array(x_signal[start:end], dtype=np.float32)
        csv_batch = np.array(x_csv[start:end], dtype=np.float32)
        batch_prediction = model([signal_batch, csv_batch], training=False).numpy()
        predictions.append(batch_prediction)

        # تحرير الذاكرة بعد كل batch
        del signal_batch, csv_batch
        gc.collect()

    if not predictions:
        return np.empty((0, model.output_shape[-1]), dtype=np.float32)

    return np.vstack(predictions).astype(np.float32)




def evaluate_multimodal_predictions(y_true, y_score, split_name, threshold=METRIC_THRESHOLD):
    y_true = np.array(y_true, dtype=np.float32)
    y_score = np.array(y_score, dtype=np.float32)
    threshold_array = np.array(threshold, dtype=np.float32)
    y_pred = (y_score >= threshold_array).astype(np.float32)

    return {
        "split": split_name,
        "threshold": float(threshold_array.mean()) if threshold_array.ndim else float(threshold_array),
        "threshold_kind": "per_class" if threshold_array.ndim else "global",
        "binary_accuracy": float((y_pred == y_true).mean()),
        "f1_micro": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "auc_micro": safe_multilabel_auc(y_true, y_score, average="micro"),
        "auc_macro": safe_multilabel_auc(y_true, y_score, average="macro"),
    }


def tune_global_threshold(y_true, y_score, grid=THRESHOLD_SEARCH_GRID, optimize_metric="f1_macro"):
    best_metrics = None
    for threshold in np.array(grid, dtype=np.float32):
        metrics = evaluate_multimodal_predictions(y_true, y_score, "validation", threshold=float(threshold))
        if best_metrics is None or metrics[optimize_metric] > best_metrics[optimize_metric]:
            best_metrics = metrics
    return best_metrics


def tune_per_class_thresholds(y_true, y_score, class_names=None, grid=THRESHOLD_SEARCH_GRID):
    y_true = np.array(y_true, dtype=np.float32)
    y_score = np.array(y_score, dtype=np.float32)
    thresholds = np.full(y_true.shape[1], METRIC_THRESHOLD, dtype=np.float32)
    rows = []

    for idx in range(y_true.shape[1]):
        best_threshold = float(METRIC_THRESHOLD)
        best_f1 = -1.0
        label_true = y_true[:, idx]
        label_score = y_score[:, idx]

        for threshold in np.array(grid, dtype=np.float32):
            label_pred = (label_score >= threshold).astype(np.float32)
            label_f1 = float(f1_score(label_true, label_pred, zero_division=0))
            if label_f1 > best_f1:
                best_f1 = label_f1
                best_threshold = float(threshold)

        thresholds[idx] = best_threshold
        label_name = class_names[idx] if class_names is not None else idx
        rows.append({
            "label": str(label_name),
            "support": int(label_true.sum()),
            "threshold": best_threshold,
            "best_label_f1": best_f1,
        })

    thresholds_df = pd.DataFrame(rows).sort_values(["support", "threshold", "label"], ascending=[False, True, True]).reset_index(drop=True)
    return thresholds, thresholds_df


In [13]:
history_df = summarize_history(history)


Last epoch metrics:
  loss  auc_roc  pr_auc  binary_accuracy  precision  recall  val_loss  val_auc_roc  val_pr_auc  val_binary_accuracy  val_precision  val_recall  val_f1_macro  lr
1.5859   0.9352  0.3374           0.9793     0.7299  0.7564    2.0093       0.9045      0.3148               0.9762         0.6872      0.7204        0.2666 0.0
Best validation AUC epoch: 27
Best validation AUC: 0.9056


In [14]:
print("=" * 60)
print("🔍 Step 1: Generating predictions on validation set...")
print("=" * 60)
val_score = batched_multimodal_predict(model, X_val_signal, X_val_csv)
gc.collect()

print("\n" + "=" * 60)
print("🔍 Step 2: Evaluating with FIXED threshold (0.5)...")
print("=" * 60)
fixed_metrics = evaluate_multimodal_predictions(y_val, val_score, "validation_fixed_0.5", threshold=METRIC_THRESHOLD)
print(pd.DataFrame([fixed_metrics]).set_index("split").round(4).to_string())

print("\n" + "=" * 60)
print("🔍 Step 3: Tuning GLOBAL threshold...")
print("=" * 60)
best_global_metrics = tune_global_threshold(y_val, val_score)
best_global_threshold = best_global_metrics["threshold"]
print(f"Best global threshold: {best_global_threshold}")
print(pd.DataFrame([best_global_metrics]).set_index("split").round(4).to_string())

print("\n" + "=" * 60)
print("🔍 Step 4: Tuning PER-CLASS thresholds...")
print("=" * 60)
per_class_thresholds, thresholds_df = tune_per_class_thresholds(
    y_val, val_score, class_names=mlb.classes_
)
print("\nPer-class thresholds:")
print(thresholds_df.to_string(index=False))

per_class_val_metrics = evaluate_multimodal_predictions(
    y_val, val_score, "validation_per_class", threshold=per_class_thresholds
)
print("\nValidation metrics with per-class thresholds:")
print(pd.DataFrame([per_class_val_metrics]).set_index("split").round(4).to_string())

print("\n" + "=" * 60)
print("🔍 Step 5: Generating predictions on TEST set...")
print("=" * 60)
test_score = batched_multimodal_predict(model, X_test_signal, X_test_csv)
gc.collect()

print("\n" + "=" * 60)
print("📊 Step 6: Final TEST evaluation — comparison")
print("=" * 60)

# مقارنة: threshold ثابت vs global tuned vs per-class
test_fixed = evaluate_multimodal_predictions(y_test, test_score, "test_fixed_0.5", threshold=0.5)
test_global = evaluate_multimodal_predictions(y_test, test_score, f"test_global_{best_global_threshold}", threshold=best_global_threshold)
test_per_class = evaluate_multimodal_predictions(y_test, test_score, "test_per_class", threshold=per_class_thresholds)

comparison_df = pd.DataFrame([test_fixed, test_global, test_per_class]).set_index("split").round(4)
print("\n🏆 Test Results Comparison:")
print(comparison_df.to_string())
print("\n✅ Per-class thresholds should give the BEST F1 scores!")

# حفظ الـ per-class thresholds للاستخدام في الـ production
np.save(BASE_DIR / "per_class_thresholds.npy", per_class_thresholds)
print(f"\n💾 Per-class thresholds saved to: {BASE_DIR / 'per_class_thresholds.npy'}")


🔍 Step 1: Generating predictions on validation set...

🔍 Step 2: Evaluating with FIXED threshold (0.5)...
                      threshold threshold_kind  binary_accuracy  f1_micro  f1_macro  f1_weighted  auc_micro  auc_macro
split                                                                                                                 
validation_fixed_0.5        0.5         global           0.9767     0.705    0.2577       0.6612     0.9721     0.9019

🔍 Step 3: Tuning GLOBAL threshold...
Best global threshold: 0.3799999952316284
            threshold threshold_kind  binary_accuracy  f1_micro  f1_macro  f1_weighted  auc_micro  auc_macro
split                                                                                                       
validation       0.38         global           0.9503    0.5722    0.2795       0.6305     0.9721     0.9019

🔍 Step 4: Tuning PER-CLASS thresholds...

Per-class thresholds:
  label  support  threshold  best_label_f1
     SR     1670      